In [ ]:
import crosswalk
import pandas as pd, numpy as np

from vivarium_gates_mncnh.constants.paths import PPH_CROSSWALK_PARAMETERS_CSV

In [2]:
import pickle

# Open the file in read-binary mode
with open('results.pkl', 'rb') as file:
    # Load the object from the file
    model = pickle.load(file)

In [3]:
input_df = pd.DataFrame({
    'dorm': 'postpartum_hem',
    'to_dorm': 'all_hem',
    'age': [12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5], # midpoints of age groups
    'mean': 1.0,
    'se': 0.0,
})
crosswalked = input_df.join(model.adjust_orig_vals(
    input_df,
    orig_dorms='dorm',
    orig_vals_mean='mean',
    orig_vals_se='se',
    ref_dorms='to_dorm',
))
crosswalked['age_start'] = crosswalked['age'] - 2.5
crosswalked['age_end'] = crosswalked['age'] + 2.5
crosswalked['sex'] = 'female'
crosswalked = crosswalked.set_index(['age_start', 'age_end', 'sex'])[['pred_diff_mean', 'pred_diff_sd']]
crosswalked

,,,pred_diff_mean,pred_diff_sd
age_start,age_end,sex,,
10.0,15.0,female,-0.829067,0.314744
15.0,20.0,female,-0.729488,0.188411
20.0,25.0,female,-0.636943,0.181245
25.0,30.0,female,-0.593641,0.213088
30.0,35.0,female,-0.601271,0.229882
35.0,40.0,female,-0.627765,0.210128
40.0,45.0,female,-0.666090,0.311027
45.0,50.0,female,-0.706105,0.549234
50.0,55.0,female,-0.679698,0.165991


In [ ]:
crosswalked.to_csv(PPH_CROSSWALK_PARAMETERS_CSV)
print(f"Saved crosswalk parameters to {PPH_CROSSWALK_PARAMETERS_CSV}")

In [4]:
crosswalked_draws = {}
np.random.seed(1234) # for reproducibility

for i in range(500):
    crosswalked_draws[f'draw_{i}'] = np.exp(np.random.normal(loc=crosswalked['pred_diff_mean'], scale=crosswalked['pred_diff_sd']))

crosswalked_draws = pd.DataFrame(crosswalked_draws, index=crosswalked.index)
crosswalked_draws

,,,draw_0,draw_1,draw_2,draw_3,draw_4,draw_5,draw_6,draw_7,draw_8,draw_9,...,draw_490,draw_491,draw_492,draw_493,draw_494,draw_495,draw_496,draw_497,draw_498,draw_499
age_start,age_end,sex,,,,,,,,,,,,,,,,,,,,,
10.0,15.0,female,0.506270,0.215469,0.661503,0.246359,0.453929,0.329084,0.367283,0.408734,0.544757,0.142180,...,0.412766,0.420886,0.350953,0.368400,0.339784,0.362970,0.527935,0.627956,0.234121,0.368276
15.0,20.0,female,0.385243,0.598811,0.360255,0.465805,0.453706,0.469888,0.469195,0.585840,0.532138,0.618426,...,0.370604,0.574632,0.496999,0.378506,0.513328,0.608406,0.447112,0.470329,0.443353,0.558702
20.0,25.0,female,0.685730,0.633081,0.509833,0.640818,0.616071,0.530663,0.563956,0.342318,0.447167,0.543742,...,0.404598,0.587146,0.497015,0.824999,0.615467,0.446266,0.601419,0.604152,0.668940,0.579658
25.0,30.0,female,0.516715,0.676719,0.480264,0.507420,0.919289,0.648774,0.548149,0.851341,0.847222,0.572020,...,0.442085,1.014111,0.627112,0.545749,0.496276,0.545960,0.506852,0.769908,0.493628,0.535537
30.0,35.0,female,0.464441,0.344411,0.573036,0.592325,0.557801,0.575921,0.624242,0.421498,0.577472,0.496515,...,0.609016,0.690857,0.698918,0.657953,0.521838,0.617312,0.456958,0.273985,0.547138,0.594498
35.0,40.0,female,0.643169,0.497597,0.599612,0.665218,0.473885,0.636961,0.738614,0.558086,0.418964,0.627181,...,0.713776,0.498851,0.409911,0.588061,0.537661,0.891853,0.368383,0.562449,0.498583,0.402234
40.0,45.0,female,0.671168,0.514052,0.774057,0.711219,0.519520,0.327661,0.379423,0.639606,0.625297,0.697848,...,0.381632,0.868831,0.467001,0.938580,0.660080,0.229052,0.566320,0.450022,0.442467,0.614530
45.0,50.0,female,0.347947,0.616673,0.381416,0.793166,0.157906,0.228524,0.474857,0.320622,0.504391,0.572723,...,0.581395,0.601591,0.519029,0.501346,0.947392,0.373881,0.793935,0.233137,0.556597,1.116261
50.0,55.0,female,0.508092,0.531681,0.566906,0.496603,0.528049,0.498351,0.533350,0.547167,0.547379,0.638495,...,0.585443,0.600861,0.526393,0.399118,0.569915,0.561941,0.592432,0.544024,0.416937,0.547632


In [5]:
crosswalked_draws.T.describe().T

,,,count,mean,std,min,25%,50%,75%,max
age_start,age_end,sex,,,,,,,,
10.0,15.0,female,500.0,0.456544,0.147972,0.142180,0.349457,0.430580,0.540493,0.934286
15.0,20.0,female,500.0,0.492554,0.095327,0.260028,0.428461,0.483278,0.547358,0.866235
20.0,25.0,female,500.0,0.542970,0.100214,0.307869,0.464746,0.531175,0.604486,0.931987
25.0,30.0,female,500.0,0.572452,0.122926,0.312880,0.487525,0.561345,0.642274,1.097050
30.0,35.0,female,500.0,0.557581,0.127154,0.260645,0.466067,0.542612,0.631170,1.051420
35.0,40.0,female,500.0,0.542861,0.111122,0.236159,0.462430,0.537227,0.611408,0.912310
40.0,45.0,female,500.0,0.551794,0.167144,0.229052,0.435363,0.533488,0.648378,1.168744
45.0,50.0,female,500.0,0.574061,0.310220,0.118698,0.351883,0.504618,0.717799,2.018037
50.0,55.0,female,500.0,0.517758,0.081970,0.342011,0.460061,0.507424,0.569768,0.777408
